# Intent classifier tự host: train bằng tiếng Anh, dịch câu tiếng Việt khi chạy

**Mục tiêu:** thay LLM gọi API bằng model tự host. Bộ phân loại chỉ học **tiếng Anh**. Khi chạy thật, câu tiếng Việt của khách được dịch sang tiếng Anh rồi mới phân loại:

```
Câu khách ──► nhận diện ngôn ngữ ──► (nếu tiếng Việt) model dịch vi→en ──► bộ phân loại tiếng Anh ──► nhãn intent
```

Đầu ra là một nhãn (ví dụ `track_order`) nên **không cần dịch ngược**.

**Hai loại model trong notebook**
- *Model phân loại* (encoder, được fine-tune ở đây): RoBERTa, DistilRoBERTa, BERT, DeBERTa-v3. Chúng chỉ đọc câu và chọn nhãn, không dịch được.
- *Model dịch* (seq2seq, dùng nguyên bản, không train): `vinai-translate-vi2en-v2` (chất lượng cao, nặng) và `opus-mt-vi-en` (nhẹ, nhanh hơn trên CPU).

**Dữ liệu**
- `dataset_en.jsonl`: 1000 câu, 14 nhãn. Mỗi dòng có `text` (bản tiếng Anh) và `text_vi` (câu gốc tiếng Việt).
- Bitext tiếng Anh (HuggingFace), gộp 46 intent về 14 nhãn, chỉ thêm vào tập train.
- *(tuỳ chọn)* `benchmark_scenarios.json`: tập test phụ.

**Câu hỏi notebook trả lời**
1. Độ chính xác **thật** khi câu tiếng Việt đi qua model dịch là bao nhiêu? (cột `real_*`)
2. Bước dịch làm mất bao nhiêu điểm so với bản dịch chuẩn? (cột `gold_en_acc` là mức trần)
3. Dịch rồi phân loại có hơn phân loại thẳng tiếng Việt không? (dòng `xlmr_vi_direct` là model đối chứng, học thẳng câu tiếng Việt)
4. Tổng độ trễ CPU (dịch + phân loại) có chấp nhận được không?

**Thiết lập Kaggle:** Accelerator **GPU T4 x2**, **Internet On**, rồi Run All. Dữ liệu tự lấy bằng cách clone repo (nhánh `REPO_BRANCH`), hoặc upload `dataset_en.jsonl` qua *Add Data*.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "datasets", "sentencepiece", "sacremoses", "protobuf"], check=False)

In [ ]:
import gc, glob, inspect, json, os, queue, random, re, threading, time, traceback, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,
                          get_linear_schedule_with_warmup)

N_GPU = torch.cuda.device_count()
GPUS = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
print("devices:", GPUS, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs").resolve()
WORK.mkdir(parents=True, exist_ok=True)

def find(pattern):
    return sorted(set(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(pattern)
                      + glob.glob(f"data/intent_eval/{pattern}") + glob.glob(f"data/{pattern}")))

### Lấy dữ liệu

Notebook tìm `dataset_en.jsonl` theo thứ tự: file đã upload trong `/kaggle/input/`, rồi thư mục repo hiện tại. Nếu chưa thấy thì **clone repo GitHub** nhánh `REPO_BRANCH` (cần Internet).

In [ ]:
REPO_URL = "https://github.com/DunnTapCodee/E-commerce-Support-Copilot.git"
REPO_BRANCH = "hung"

if not find("dataset_en.jsonl"):
    repo = WORK / "repo"
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    print("repo commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

print("dataset:", find("dataset_en.jsonl"))
print("benchmark:", find("benchmark_scenarios.json"))
assert find("dataset_en.jsonl"), "Không tìm thấy dataset_en.jsonl: upload qua Add Data hoặc kiểm tra REPO_BRANCH."

## 1. Cấu hình

In [ ]:
MODELS = {
    # tên ngắn: (HF id, ngôn ngữ đầu vào của model)
    "roberta":        ("roberta-base", "en"),
    "distilroberta":  ("distilroberta-base", "en"),
    "bert":           ("bert-base-uncased", "en"),
    "deberta_v3":     ("microsoft/deberta-v3-base", "en"),
    "xlmr_vi_direct": ("xlm-roberta-base", "vi"),   # ĐỐI CHỨNG: học thẳng câu tiếng Việt, không qua dịch
}
RUN = ["roberta", "distilroberta", "bert", "deberta_v3", "xlmr_vi_direct"]   # bỏ bớt để chạy nhanh
SEEDS = [42]                 # [42, 7, 2024] để đo độ dao động (thời gian x3)

MT_MODELS = {                # model dịch vi -> en, dùng nguyên bản
    "vinai": "vinai/vinai-translate-vi2en-v2",
    "opus":  "Helsinki-NLP/opus-mt-vi-en",
}
PRIMARY_MT = "vinai"         # model dịch dùng cho cột real_*, cho hiệu chỉnh ngưỡng và cho bản export
MT_BEAMS = 4
TRAIN_WITH_MT = True         # thêm bản dịch máy của các câu train, để bộ phân loại quen với lỗi của model dịch

USE_BITEXT = True
EN_PER_LABEL = 100           # số câu Bitext tối đa mỗi nhãn, chia đều cho các intent gốc được gộp vào nhãn đó

EPOCHS, LR, BATCH, MAX_LEN, WARMUP, WD = 10, 3e-5, 16, 96, 0.1, 0.01
PATIENCE = 3
TARGET_ACCEPTED_ACC = 0.95   # ngưỡng confidence: câu được giữ phải đúng >= 95% trên val

LABELS = ["track_order", "cancel_order", "delivery_time", "delivery_issue", "shipping_costs",
          "return_policy", "return_product", "refund_policy", "request_refund", "refund_status",
          "payment_issue", "human_agent", "other_ecommerce", "out_of_scope"]
L2I = {l: i for i, l in enumerate(LABELS)}

## 2. Dữ liệu và nhận diện ngôn ngữ

Bộ nhận diện ngôn ngữ dùng cùng cách với `LanguageRouter` của `intent_service`: có dấu tiếng Việt thì là tiếng Việt; không dấu thì đếm tỷ lệ từ tiếng Việt không dấu quen thuộc. Câu nào bị nhận nhầm sẽ đi sai đường (không được dịch), và lỗi đó được tính vào kết quả `real_*`.

In [ ]:
df = pd.read_json(find("dataset_en.jsonl")[0], lines=True).rename(columns={"text": "en", "text_vi": "vi"})
assert set(df.intent) <= set(LABELS), set(df.intent) - set(LABELS)

VI_CHARS = re.compile(r"[ăâđêôơưàáạảãằắặẳẵầấậẩẫèéẹẻẽềếệểễìíịỉĩòóọỏõồốộổỗờớợởỡùúụủũừứựửữỳýỵỷỹ]", re.I)
VI_NOTONE = frozenset(("toi muon huy don hang hoan tien phi bao nhieu giao chua khong minh nay giup voi duoc cho nhan "
                       "vien chinh sach doi tra san pham kiem tinh trang ko dc oi roi nhe quen khau khoan dia gui dau vay "
                       "thanh toan bi loi qua thi nhan lam lay lai cua dang buoc nao mua ny dh").split())

def is_vi(text):
    if VI_CHARS.search(text): return True
    toks = re.findall(r"[a-z]+", text.lower())
    return bool(toks) and sum(t in VI_NOTONE for t in toks) / len(toks) >= 0.2

df["is_vi"] = df.vi.map(is_vi)
truth = df.vi != df.en          # câu vốn là tiếng Anh thì bản dịch giữ nguyên
print(f"{len(df)} câu | nhận diện ngôn ngữ đúng {(df.is_vi == truth).mean():.1%} "
      f"| tiếng Việt bị coi là tiếng Anh: {int((truth & ~df.is_vi).sum())} | ngược lại: {int((~truth & df.is_vi).sum())}")

## 3. Dịch máy vi→en

Dịch **toàn bộ** câu tiếng Việt bằng từng model dịch (kết quả lưu cache trong `/kaggle/working`, chạy lại không dịch lại). Sau đó đo độ trễ của model dịch trên **CPU**, vì đó là chi phí thật khi tự host không có GPU.

In [ ]:
MT_CPU_MS = {}

def load_mt(name, dev):
    hf = MT_MODELS[name]
    tok = AutoTokenizer.from_pretrained(hf, **({"src_lang": "vi_VN"} if "vinai" in hf else {}))
    return tok, AutoModelForSeq2SeqLM.from_pretrained(hf).to(dev).eval()

def translate(texts, tok, model, name, dev, bs=32):
    extra = {"decoder_start_token_id": tok.convert_tokens_to_ids("en_XX")} if "vinai" in MT_MODELS[name] else {}
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
        with torch.inference_mode(), torch.autocast("cuda", enabled=dev.startswith("cuda")):
            ids = model.generate(**enc, num_beams=MT_BEAMS, early_stopping=True, max_new_tokens=128, **extra)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out

def mt_all(name, texts):
    cache_p = WORK / f"mt_cache_{name}.json"
    cache = json.load(open(cache_p, encoding="utf-8")) if cache_p.exists() else {}
    todo = sorted({t for t in texts if t not in cache}, key=len)
    if todo:
        tok, model = load_mt(name, GPUS[0]); t0 = time.time()
        cache.update(zip(todo, translate(todo, tok, model, name, GPUS[0])))
        json.dump(cache, open(cache_p, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"[{name}] dịch {len(todo)} câu trong {time.time() - t0:.0f}s trên {GPUS[0]}")
        # độ trễ CPU: từng câu một, giống lúc chạy thật
        m = model.to("cpu").float(); sample = todo[len(todo) // 2: len(todo) // 2 + 8]
        translate(sample[:1], tok, m, name, "cpu", bs=1); t0 = time.perf_counter()
        for s in sample: translate([s], tok, m, name, "cpu", bs=1)
        MT_CPU_MS[name] = (time.perf_counter() - t0) / len(sample) * 1000
        print(f"[{name}] CPU: {MT_CPU_MS[name]:.0f} ms/câu (beam {MT_BEAMS})")
        del model, m; gc.collect(); torch.cuda.empty_cache()
    return [cache[t] for t in texts]

def to_english(frame, name):
    # đường đi thật: câu được nhận là tiếng Việt thì dịch, còn lại giữ nguyên
    out = frame.vi.copy(); mask = frame.vi.map(is_vi)
    out[mask] = mt_all(name, frame.vi[mask].tolist())
    return out

for name in MT_MODELS:
    df[f"mt_{name}"] = to_english(df, name)
pd.set_option("display.max_colwidth", 90)
df.sample(8, random_state=1)[["vi", "en"] + [f"mt_{n}" for n in MT_MODELS]]

In [ ]:
# Test 20% cố định: cùng cách chia và random_state với notebook TF-IDF, nên id tập test trùng nhau.
# Chia theo dòng (mỗi id một dòng), nên bản tiếng Việt, bản dịch chuẩn và bản dịch máy của một câu luôn cùng phía.
trainval, test_df = train_test_split(df, test_size=0.2, stratify=df.intent, random_state=42)
train_df, val_df = train_test_split(trainval, test_size=0.125, stratify=trainval.intent, random_state=42)
y_val, y_test = val_df.intent.map(L2I).to_numpy(), test_df.intent.map(L2I).to_numpy()
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

bench_df = pd.DataFrame()
if find("benchmark_scenarios.json"):
    b = pd.DataFrame(json.load(open(find("benchmark_scenarios.json")[0], encoding="utf-8")))
    bench_df = pd.DataFrame({"vi": b["query"], "intent": b["expected_intent"].replace({"warranty": "other_ecommerce"})})
    bench_df = bench_df[bench_df.intent.isin(LABELS)].drop_duplicates("vi").reset_index(drop=True)
    bench_df["mt"] = to_english(bench_df, PRIMARY_MT)
    print("benchmark:", len(bench_df), "câu (đã bỏ trùng)")

## 4. Tập train tiếng Anh

Gồm 3 phần, tất cả là tiếng Anh:
1. Bản dịch chuẩn của các câu train (`dataset_en.jsonl`).
2. *(nếu `TRAIN_WITH_MT`)* Bản dịch máy của chính các câu train đó. Khi chạy thật, bộ phân loại nhận câu do model dịch sinh ra, nên cho nó học trước kiểu câu này.
3. *(nếu `USE_BITEXT`)* Bitext. `BITEXT_MAP` gộp 46 intent về 14 nhãn; các intent mơ hồ (`wrong_item`, `missing_item`, `damaged_delivery`, `product_issue`, `pay`) bị bỏ. Mỗi nhãn lấy tối đa `EN_PER_LABEL` câu, chia đều cho các intent gốc.

Tập val và test **không** có mặt trong bất kỳ phần nào ở trên.

In [ ]:
BITEXT_MAP = {
    "track_order": "track_order", "track_delivery": "track_order", "cancel_order": "cancel_order",
    "delivery_time": "delivery_time", "delivery_issue": "delivery_issue", "shipping_costs": "shipping_costs",
    "return_policy": "return_policy", "return_product": "return_product", "return_product_online": "return_product",
    "return_product_in_store": "return_product", "refund_policy": "refund_policy", "request_refund": "request_refund",
    "refund_status": "refund_status", "payment_issue": "payment_issue", "human_agent": "human_agent",
    "customer_service": "human_agent",
    **{k: "other_ecommerce" for k in ["add_product", "availability", "availability_in_store", "availability_online",
        "change_account", "change_order", "close_account", "open_account", "order_history", "payment_methods",
        "product_information", "recover_password", "remove_product", "request_invoice", "sales_period",
        "store_location", "store_opening_hours", "submit_feedback", "submit_product_feedback",
        "submit_product_idea", "use_app", "technical_issue", "exchange_product", "exchange_product_in_store",
        "request_right_to_rectification"]},
}
FILL = {"order number": "ORD-48213", "person name": "Lan", "store location": "District 7",
        "product name": "headphones", "refund amount": "350k VND"}

def fill_placeholders(t):
    return re.sub(r"\{\{\s*([^}]+?)\s*\}\}", lambda m: FILL.get(m.group(1).lower(), m.group(1)), t)

def preprocess(t):   # giống preprocess() của intent_service: NFC, gom khoảng trắng, hạ chữ thường
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(t))).strip().lower()

parts = [pd.DataFrame({"text": train_df.en, "intent": train_df.intent, "src": "gold_en"})]
if TRAIN_WITH_MT:
    for name in MT_MODELS:
        parts.append(pd.DataFrame({"text": train_df[f"mt_{name}"], "intent": train_df.intent, "src": f"mt_{name}"}))
if USE_BITEXT:
    from datasets import load_dataset
    bx = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset", split="train").to_pandas()
    text_col = next(c for c in ["instruction", "utterance", "text"] if c in bx.columns)
    bx = pd.DataFrame({"text": bx[text_col].map(fill_placeholders), "src_intent": bx["intent"],
                       "intent": bx["intent"].map(BITEXT_MAP)}).dropna(subset=["intent"]).drop_duplicates("text")
    bx = bx.sample(frac=1, random_state=42)
    quota = EN_PER_LABEL / bx.groupby("intent").src_intent.transform("nunique")
    bx = bx[bx.groupby(["intent", "src_intent"]).cumcount() < quota.clip(lower=1)]
    parts.append(bx.assign(src="bitext")[["text", "intent", "src"]])

train_en = pd.concat(parts, ignore_index=True)
train_en = train_en[~train_en.text.map(preprocess).duplicated()].reset_index(drop=True)
print("Tập train tiếng Anh:", len(train_en))
display(pd.crosstab(train_en.intent, train_en.src).reindex(LABELS))

def xy(texts, intents): return list(texts), pd.Series(list(intents)).map(L2I).to_numpy()
DATA = {
    # "real" = đường đi thật của câu khách. Với model tiếng Anh: câu Việt qua model dịch chính.
    "en": {"train": xy(train_en.text, train_en.intent), "val": xy(val_df[f"mt_{PRIMARY_MT}"], val_df.intent),
           "real": list(test_df[f"mt_{PRIMARY_MT}"]),
           "extra": {"gold_en": list(test_df.en), **{f"mt_{n}": list(test_df[f"mt_{n}"]) for n in MT_MODELS}},
           "bench": xy(bench_df.mt, bench_df.intent) if len(bench_df) else None},
    # đối chứng: model đa ngôn ngữ đọc thẳng câu tiếng Việt
    "vi": {"train": xy(train_df.vi, train_df.intent), "val": xy(val_df.vi, val_df.intent),
           "real": list(test_df.vi), "extra": {},
           "bench": xy(bench_df.vi, bench_df.intent) if len(bench_df) else None},
}

## 5. Vòng train (PyTorch thuần, fp16 khi có GPU)

Mọi hàm nhận tham số `dev` (`cuda:0`, `cuda:1`, `cpu`) để train nhiều model song song, mỗi GPU một model.

In [ ]:
SEED_LOCK = threading.Lock()   # khởi tạo model dưới khoá để seed không bị luồng khác chen vào

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def batches(tok, texts, ys, bs, dev, shuffle=False, seed=0):
    idx = np.arange(len(texts))
    if shuffle: np.random.default_rng(seed).shuffle(idx)
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        enc = tok([preprocess(texts[k]) for k in j], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        if ys is not None: enc["labels"] = torch.tensor(ys[j])
        yield {k: v.to(dev) for k, v in enc.items()}

@torch.no_grad()
def logits_of(model, tok, texts, dev):
    model.eval(); out = []
    for b in batches(tok, texts, None, 64, dev):
        with torch.autocast("cuda", enabled=dev.startswith("cuda")):
            out.append(model(**b).logits.float().cpu())
    return torch.cat(out).numpy()

def train_one(key, seed, dev):
    hf_id, mode = MODELS[key]
    (tr_x, tr_y), (va_x, va_y) = DATA[mode]["train"], DATA[mode]["val"]
    gpu = dev.startswith("cuda")
    with SEED_LOCK:
        set_seed(seed)
        tok = AutoTokenizer.from_pretrained(hf_id)
        model = AutoModelForSequenceClassification.from_pretrained(
            hf_id, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
    model.to(dev)
    steps = EPOCHS * ((len(tr_x) + BATCH - 1) // BATCH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=gpu)
    best, best_state, bad, hist = -1, None, 0, []
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for b in batches(tok, tr_x, tr_y, BATCH, dev, shuffle=True, seed=seed + ep):
            with torch.autocast("cuda", enabled=gpu):
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            tot += loss.item(); n += 1
        vf1 = f1_score(va_y, logits_of(model, tok, va_x, dev).argmax(1), average="macro")
        hist.append({"epoch": ep + 1, "loss": tot / n, "val_f1": vf1})
        print(f"  [{dev} {key} s{seed}] epoch {ep+1}: loss {tot/n:.3f}  val macro-F1 {vf1:.4f}")
        if vf1 > best:
            best, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, tok, time.time() - t0, hist

## 6. Hiệu chỉnh confidence và chọn ngưỡng

- **Temperature scaling:** tìm hệ số `T` để `softmax(logits / T)` sát với xác suất đúng thật.
- **ECE:** độ lệch trung bình giữa confidence model báo và tỷ lệ đúng thực tế. Càng thấp càng tốt.
- **Ngưỡng:** confidence nhỏ nhất sao cho các câu được giữ lại đúng ≥ 95%. Câu dưới ngưỡng chuyển cho nhân viên. **Coverage** là tỷ lệ câu model tự xử lý.

Cả `T` và ngưỡng được chọn trên **tập val đi theo đường thật** (câu Việt qua model dịch chính), không dùng tập test.

In [ ]:
def ece(y, p, bins=15):
    conf, correct, tot = p.max(1), p.argmax(1) == y, 0.0
    for i in range(bins):
        m = (conf > i / bins) & (conf <= (i + 1) / bins)
        if m.any(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

def fit_temperature(logits, y):
    nll = lambda T: -np.log(np.clip(softmax(logits / T, 1)[np.arange(len(y)), y], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x)

def pick_threshold(p, y):
    conf, ok = p.max(1), p.argmax(1) == y
    for t in np.round(np.arange(0.30, 0.96, 0.01), 2):
        keep = conf >= t
        if keep.any() and ok[keep].mean() >= TARGET_ACCEPTED_ACC: return float(t)
    return None

## 7. Chạy so sánh, song song trên nhiều GPU

Mỗi cặp (model, seed) là một job trong hàng đợi. Mỗi GPU có một luồng riêng lấy job ra train; GPU nào xong trước thì nhận việc tiếp. Máy có 1 GPU hoặc chỉ có CPU thì chạy tuần tự.

Các cột kết quả:
- `real_acc`, `real_macro_f1`: đường đi thật. Với model tiếng Anh là câu Việt → model dịch chính → phân loại. Với `xlmr_vi_direct` là phân loại thẳng câu Việt.
- `gold_en_acc`: phân loại trên bản dịch chuẩn. Đây là mức trần nếu model dịch hoàn hảo.
- `mt_<tên>_acc`: đường đi thật với từng model dịch.

Log của 2 GPU in xen kẽ nhau, mỗi dòng có tiền tố `[cuda:0 …]` hoặc `[cuda:1 …]`.

In [ ]:
jobs = queue.Queue()
SIZE = {"xlmr_vi_direct": 278, "deberta_v3": 184, "roberta": 125, "bert": 110, "distilroberta": 82}
for key in sorted(RUN, key=lambda k: -SIZE.get(k, 150)):      # model lớn chạy trước
    for seed in SEEDS: jobs.put((key, seed))

rows, store, errors, LOCK = [], {}, [], threading.Lock()

def evaluate_job(key, seed, dev):
    D = DATA[MODELS[key][1]]
    model, tok, train_s, hist = train_one(key, seed, dev)
    val_logits = logits_of(model, tok, D["val"][0], dev)
    T = fit_temperature(val_logits, D["val"][1])
    thr = pick_threshold(softmax(val_logits / T, 1), D["val"][1])
    p_real = softmax(logits_of(model, tok, D["real"], dev) / T, 1)
    pred = p_real.argmax(1); keep = p_real.max(1) >= (thr if thr is not None else 1.01)
    r = {"model": key, "input": MODELS[key][1], "seed": seed, "gpu": dev,
         "real_acc": accuracy_score(y_test, pred), "real_macro_f1": f1_score(y_test, pred, average="macro"),
         "ece": ece(y_test, p_real), "temperature": T, "threshold": thr, "coverage": keep.mean(),
         "acc_on_kept": (pred[keep] == y_test[keep]).mean() if keep.any() else np.nan,
         "epochs": len(hist), "train_s": train_s, "params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    p_extra = {}
    for name, texts in D["extra"].items():
        p_extra[name] = softmax(logits_of(model, tok, texts, dev) / T, 1)
        r[f"{name}_acc"] = accuracy_score(y_test, p_extra[name].argmax(1))
    if D["bench"] is not None:
        r["bench_acc"] = accuracy_score(D["bench"][1], logits_of(model, tok, D["bench"][0], dev).argmax(1))
    vf1 = max(h["val_f1"] for h in hist)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model; gc.collect(); torch.cuda.empty_cache()
    with LOCK:
        rows.append(r)
        if key not in store or vf1 > store[key]["val_f1"]:
            store[key] = {"val_f1": vf1, "T": T, "thr": thr, "p_real": p_real, "p_extra": p_extra, "state": state}
    print(f"[{dev}] DONE {key} s{seed}: real acc {r['real_acc']:.4f} | macro-F1 {r['real_macro_f1']:.4f} | {train_s/60:.1f} phút")

def worker(dev):
    while True:
        try: key, seed = jobs.get_nowait()
        except queue.Empty: return
        try: evaluate_job(key, seed, dev)
        except Exception as e:      # một model lỗi không làm dừng các model khác
            with LOCK: errors.append((key, seed, dev, repr(e)))
            traceback.print_exc(); torch.cuda.empty_cache()

t0 = time.time()
threads = [threading.Thread(target=worker, args=(d,)) for d in GPUS]
for th in threads: th.start()
for th in threads: th.join()
print(f"Tổng thời gian: {(time.time() - t0) / 60:.1f} phút trên {GPUS}")
if errors: print("LỖI:", errors)

In [ ]:
# Độ trễ CPU của riêng bộ phân loại (PyTorch fp32, từng câu một), đo tuần tự sau khi train xong
clf_ms = {}
for key in store:
    hf_id, mode = MODELS[key]
    tok = AutoTokenizer.from_pretrained(hf_id)
    m = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=len(LABELS))
    m.load_state_dict(store[key]["state"]); m.eval(); sample = DATA[mode]["real"][:40]
    with torch.no_grad():
        list(logits_of(m, tok, sample[:1], "cpu")); t = time.perf_counter()
        for s in sample: logits_of(m, tok, [s], "cpu")
    clf_ms[key] = (time.perf_counter() - t) / len(sample) * 1000; del m

res = pd.DataFrame(rows)
res["clf_cpu_ms"] = res.model.map(clf_ms)
# tổng độ trễ ước tính cho một câu tiếng Việt: dịch (nếu model cần) + phân loại
res["total_cpu_ms"] = res.clf_cpu_ms + np.where(res.input == "en", MT_CPU_MS.get(PRIMARY_MT, np.nan), 0)
agg = res.drop(columns=["seed"]).groupby("model").mean(numeric_only=True)
agg["acc_std"] = res.groupby("model").real_acc.std()
agg.insert(0, "input", res.groupby("model").input.first())

### Mốc so sánh: TF-IDF + LinearSVC và LLM

- **TF-IDF + LinearSVC** là kiểu model nhánh `hung` đang dùng cho tiếng Anh. Train trên cùng tập train tiếng Anh, đánh giá trên cùng đường đi thật.
- **LLM**: nếu đã chạy `eval_llm.py` và có file `results_<model>.jsonl`, kết quả LLM trên cùng 200 câu test được thêm vào bảng.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC
tfidf = make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, preprocessor=preprocess),
                                 TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, preprocessor=preprocess)),
                      LinearSVC(C=0.5, class_weight="balanced"))
tfidf.fit(*DATA["en"]["train"])
pr = tfidf.predict(DATA["en"]["real"])
agg.loc["tfidf_svm_en", ["real_acc", "real_macro_f1", "gold_en_acc"]] = [
    accuracy_score(y_test, pr), f1_score(y_test, pr, average="macro"), accuracy_score(y_test, tfidf.predict(DATA["en"]["extra"]["gold_en"]))]
agg.loc["tfidf_svm_en", "input"] = "en"
agg.loc["tfidf_svm_en", "total_cpu_ms"] = MT_CPU_MS.get(PRIMARY_MT, np.nan)

for path in find("results_*.jsonl"):
    llm = pd.read_json(path, lines=True).set_index("id")["pred"]
    m = test_df.join(llm.rename("p"), on="id").dropna(subset=["p"])
    if len(m):
        name = "LLM:" + Path(path).stem[8:]
        agg.loc[name, ["real_acc", "real_macro_f1"]] = [accuracy_score(m.intent, m.p), f1_score(m.intent, m.p, average="macro")]
        agg.loc[name, "input"] = "vi"

agg = agg.sort_values("real_macro_f1", ascending=False)
agg.to_csv(WORK / "leaderboard.csv"); res.to_csv(WORK / "runs.csv", index=False)
print("Độ trễ model dịch trên CPU (ms/câu):", {k: round(v) for k, v in MT_CPU_MS.items()} or "chưa đo (bản dịch lấy từ cache)")
cols = ["input", "real_acc", "real_macro_f1", "gold_en_acc"] + [f"mt_{n}_acc" for n in MT_MODELS] + \
       ["bench_acc", "ece", "threshold", "coverage", "acc_on_kept", "clf_cpu_ms", "total_cpu_ms", "params_M", "train_s", "acc_std"]
agg[[c for c in cols if c in agg]].round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(agg.index, agg.real_macro_f1, label="đường thật")
if "gold_en_acc" in agg: ax[0].scatter(agg.gold_en_acc, agg.index, color="k", marker="|", s=200, label="acc với bản dịch chuẩn")
ax[0].set_xlim(max(0, agg.real_macro_f1.min() - .1), 1); ax[0].set_title("Macro-F1 trên đường đi thật"); ax[0].legend()
a = agg.dropna(subset=["total_cpu_ms"])
ax[1].scatter(a.total_cpu_ms, a.real_macro_f1)
for k, r in a.iterrows(): ax[1].annotate(k, (r.total_cpu_ms, r.real_macro_f1))
ax[1].set_xlabel("Tổng độ trễ CPU / câu tiếng Việt (ms): dịch + phân loại"); ax[1].set_ylabel("Macro-F1")
plt.tight_layout(); plt.show()

## 8. Bước dịch làm hỏng những câu nào?

Với model tiếng Anh tốt nhất: liệt kê các câu mà **bản dịch chuẩn thì đoán đúng, nhưng bản dịch máy thì đoán sai**. Đó là phần mất do model dịch. Bảng theo kiểu câu cho thấy model dịch yếu ở đâu (teencode, không dấu, giọng miền…).

In [ ]:
en_models = [k for k in agg.index if k in store and MODELS[k][1] == "en"]
BEST = en_models[0]              # model tiếng Anh tốt nhất theo real_macro_f1; có thể đặt tay, ví dụ "roberta"
s = store[BEST]
t = test_df.assign(pred_real=[LABELS[i] for i in s["p_real"].argmax(1)], conf=s["p_real"].max(1),
                   pred_gold=[LABELS[i] for i in s["p_extra"]["gold_en"].argmax(1)])
t["ok_real"], t["ok_gold"] = t.pred_real == t.intent, t.pred_gold == t.intent
print("Model:", BEST, "| T =", round(s["T"], 3), "| ngưỡng =", s["thr"])
print(f"Đúng với bản dịch chuẩn: {t.ok_gold.mean():.1%} | đúng trên đường thật ({PRIMARY_MT}): {t.ok_real.mean():.1%}")
dmg = t[t.ok_gold & ~t.ok_real]
print(f"Sai do bước dịch: {len(dmg)} / {len(t)} câu")
display(dmg[["vi", f"mt_{PRIMARY_MT}", "en", "intent", "pred_real", "style"]])
display(t.groupby("style")[["ok_gold", "ok_real"]].mean().assign(n=t.groupby("style").size()).query("n >= 3").sort_values("ok_real").head(15))

In [ ]:
print(classification_report(y_test, s["p_real"].argmax(1), labels=range(len(LABELS)), target_names=LABELS, zero_division=0))
cm = confusion_matrix(y_test, s["p_real"].argmax(1), labels=range(len(LABELS)))
fig, ax = plt.subplots(figsize=(9, 8)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=75); ax.set_yticks(range(len(LABELS)), LABELS)
for i, j in zip(*np.nonzero(cm)): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Đúng"); plt.tight_layout(); plt.show()
display(t.loc[~t.ok_real, ["vi", f"mt_{PRIMARY_MT}", "intent", "pred_real", "conf"]].sort_values("conf", ascending=False))

## 9. Export bộ phân loại sang ONNX int8

Bộ phân loại được lượng tử hoá xuống int8 để chạy bằng `onnxruntime` trên CPU. `meta.json` theo định dạng `OnnxTransformerBackend` của `intent_service` (nhánh `refine`), thêm mục `input_pipeline` ghi model dịch đã dùng.

**Cần sửa code service trước khi dùng:** `predictor.py` hiện chưa có bước dịch. Với câu tiếng Việt, phải gọi model dịch vi→en rồi mới đưa vào backend `en` này. Model dịch **không** được export ở đây; `meta.json` chỉ ghi tên để service tự tải.

In [ ]:
EXPORT = BEST
hf_id = MODELS[EXPORT][0]
tok = AutoTokenizer.from_pretrained(hf_id)
model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
model.load_state_dict(store[EXPORT]["state"]); model.eval().float()

out = WORK / "intent_artifact" / "en"; out.mkdir(parents=True, exist_ok=True)
dummy = tok([preprocess("where is my order")], return_tensors="pt")
inputs = [k for k in ["input_ids", "attention_mask", "token_type_ids"] if k in dummy]
extra = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}   # torch mới mặc định dùng dynamo
torch.onnx.export(model, tuple(dummy[k] for k in inputs), str(out / "model.onnx"), input_names=inputs,
                  output_names=["logits"], opset_version=17,
                  dynamic_axes={**{k: {0: "batch", 1: "seq"} for k in inputs}, "logits": {0: "batch"}}, **extra)
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(str(out / "model.onnx"), str(out / "model_int8.onnx"), weight_type=QuantType.QInt8)
(out / "model.onnx").unlink()
tok.save_pretrained(out)

import onnxruntime as ort
sess = ort.InferenceSession(str(out / "model_int8.onnx"), providers=["CPUExecutionProvider"])
names = {i.name for i in sess.get_inputs()}
T = store[EXPORT]["T"]
def onnx_proba(text_en):
    enc = tok(preprocess(text_en), return_tensors="np", truncation=True, max_length=MAX_LEN)
    return softmax(sess.run(None, {k: enc[k].astype(np.int64) for k in names})[0][0] / T)

real_texts = DATA["en"]["real"]
t0 = time.perf_counter(); p_onnx = np.stack([onnx_proba(x) for x in real_texts]); onnx_ms = (time.perf_counter() - t0) / len(real_texts) * 1000
agree = (p_onnx.argmax(1) == store[EXPORT]["p_real"].argmax(1)).mean()
onnx_acc = accuracy_score(y_test, p_onnx.argmax(1))
print(f"ONNX int8: {onnx_ms:.1f} ms/câu | trùng dự đoán với PyTorch {agree:.1%} | real acc {onnx_acc:.4f}")
print(f"Kích thước: {sum(f.stat().st_size for f in out.iterdir()) / 1e6:.0f} MB")

In [ ]:
import importlib.metadata as im
meta = {"backend": "onnx_transformer", "method": f"ft-{EXPORT}-en", "base_model": hf_id, "lang": "en",
        "classes": LABELS, "max_length": MAX_LEN, "temperature": store[EXPORT]["T"],
        "threshold": store[EXPORT]["thr"], "needs_word_segmentation": False, "model_file": "model_int8.onnx",
        "input_pipeline": {"vi_to_en_model": MT_MODELS[PRIMARY_MT], "num_beams": MT_BEAMS,
                           "note": "Vietnamese queries must be machine-translated to English before this classifier."},
        "library_versions": {p: im.version(p) for p in ["transformers", "torch", "onnxruntime"]},
        "metrics": {"real_acc_onnx_int8": float(onnx_acc), "clf_latency_ms_onnx_int8": float(onnx_ms),
                    "mt_latency_ms_cpu": MT_CPU_MS.get(PRIMARY_MT),
                    **{k: float(v) for k, v in agg.loc[EXPORT].drop("input").dropna().items()}},
        "train_data": train_en.src.value_counts().to_dict()}
json.dump(meta, open(out / "meta.json", "w"), ensure_ascii=False, indent=2)
subprocess.run(["zip", "-qr", str(WORK / "intent_artifact.zip"), "intent_artifact"], cwd=WORK)
print(json.dumps(meta, ensure_ascii=False, indent=1)[:1800])

## 10. Chạy thử cả luồng: nhận diện ngôn ngữ → dịch → phân loại

In [ ]:
mt_tok, mt_model = load_mt(PRIMARY_MT, GPUS[0])
def predict(text):
    en = translate([text], mt_tok, mt_model, PRIMARY_MT, GPUS[0], bs=1)[0] if is_vi(text) else text
    p = onnx_proba(en); i = int(p.argmax())
    return en, LABELS[i], float(p[i])

thr = store[EXPORT]["thr"]
for q in ["đơn ORD-1001 tới đâu r shop", "toi bi tru tien 2 lan ma don van loi", "cho gặp nhân viên thật đi",
          "where is my refund", "hôm nay trời đẹp ghê", "bao lâu thì nhận được hàng zậy", "ship ve da nang het bao nhieu"]:
    en, lab, conf = predict(q)
    flag = "" if thr and conf >= thr else "  -> dưới ngưỡng, chuyển nhân viên"
    print(f"{q}\n   en: {en}\n   => {lab} ({conf:.2f}){flag}")

## Đọc kết quả thế nào

- So `real_acc` của model tiếng Anh tốt nhất với `xlmr_vi_direct`: nếu `xlmr_vi_direct` cao hơn hoặc ngang, thì phân loại thẳng tiếng Việt vừa chính xác hơn vừa không tốn bước dịch.
- So `real_acc` với `gold_en_acc`: khoảng cách là phần mất do model dịch. Khoảng cách lớn nghĩa là nên đổi model dịch, hoặc chuẩn hoá teencode/không dấu trước khi dịch.
- `total_cpu_ms` là chi phí mỗi câu tiếng Việt khi tự host trên CPU. Model dịch thường chiếm phần lớn con số này.
- Tập test chỉ có 200 câu: mỗi câu sai làm giảm 0.5%. Chênh lệch dưới 1–2% coi như ngang nhau; bật `SEEDS = [42, 7, 2024]` để xem độ dao động.
- Dữ liệu là câu do LLM viết và nhãn chưa được người rà. Kết quả trên tin nhắn khách thật có thể thấp hơn.